# 📚 Capítulo 4: Recursion - Teoría

## Estructuras de Datos y Algoritmos - Lic. en Sistemas

## 📋 Información del Material
- **Capítulo**: 4
- **Título**: Recursion
- **Tipo**: Material Teórico
- **Fecha**: 2026-03-05
- **Versión**: 1.0

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap04/01_Fundamentos_Recursion_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este capítulo, el estudiante será capaz de:

1. **Distinguir caso base y caso recursivo en una función recursiva**
2. **Implementar algoritmos recursivos clásicos: factorial, potencia, Fibonacci, búsqueda binaria**
3. **Trazar la pila de llamadas de una función recursiva y detectar errores comunes (stack overflow)**
4. **Clasificar un algoritmo recursivo según su tipo: lineal, binario, múltiple o de cola**

---

## 📖 Contenidos

### 🔄 Sección 1 — ¿Qué es la Recursión?
1. Definición y analogías del mundo real
2. Caso base y caso recursivo
3. Factorial: versión iterativa vs recursiva
4. Traza de ejecución y cómo piensa Python

### 📐 Sección 2 — La Pila de Llamadas (Call Stack)
1. Cómo Python gestiona las llamadas recursivas
2. Frames en la pila y variables locales
3. Stack overflow: errores comunes y cómo evitarlos
4. `sys.setrecursionlimit`

### 🔢 Sección 3 — Primeros Ejemplos Rigurosos
1. Potencia entera: potencia(x, n) recursiva
2. Suma de los primeros n enteros
3. Suma de dígitos

---


## 🔧 Configuración del Entorno

Importamos las librerías necesarias para este capítulo:

In [ ]:
# Librerías estándar de Python
import sys
import time
import math
from typing import List, Optional, Any, Union

# Librerías para visualización y análisis
import matplotlib.pyplot as plt
import numpy as np

# Configuración de matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"Python version: {sys.version}")
print(f"Entorno configurado correctamente ✅")

---

# 🔄 Sección 1: ¿Qué es la Recursión?

## 🎯 Introducción

> "Una función es *recursiva* si se define en términos de sí misma."
> — Goodrich, Tamassia & Goldwasser, Cap. 4

La recursión es uno de los **principios más poderosos** en programación: permite resolver problemas complejos descomponiéndolos en versiones más pequeñas del **mismo problema**, hasta llegar a un caso tan simple que ya no requiere más descomposición.

### Analogía del mundo real 🪞

> **Mirarte en espejo con otro espejo detrás**: verías infinitas copias de ti mismo... a menos que pongas un límite (el "caso base").

### Estructura de una función recursiva

Toda función recursiva tiene **exactamente dos partes**:

| Parte | Descripción | Ejemplo (factorial) |
|-------|-------------|---------------------|
| **Caso base** | Devuelve resultado directo, sin llamada recursiva | `if n == 0: return 1` |
| **Caso recursivo** | Reduce el problema y llama a la misma función | `return n * factorial(n-1)` |

⚠️ **Si falta el caso base → recursión infinita → Stack Overflow**


In [ ]:
# ─────────────────────────────────────────────────────────────
# Ejemplo 1: Factorial — iterativo vs recursivo
# n! = n * (n-1) * ... * 2 * 1  con  0! = 1
# ─────────────────────────────────────────────────────────────

def factorial_iterativo(n: int) -> int:
    """Complejidad: O(n) tiempo, O(1) espacio."""
    resultado = 1
    for k in range(2, n + 1):
        resultado *= k
    return resultado

def factorial_recursivo(n: int) -> int:
    """
    Caso base:    n == 0 → devuelve 1
    Caso recursivo: n > 0 → devuelve n * factorial(n-1)
    Complejidad: O(n) tiempo, O(n) espacio (pilas de llamadas)
    """
    if n == 0:           # ← caso base
        return 1
    return n * factorial_recursivo(n - 1)   # ← caso recursivo

# Demostración
print("Factorial iterativo vs recursivo")
print("─" * 40)
for n in range(8):
    iter_val = factorial_iterativo(n)
    rec_val  = factorial_recursivo(n)
    match = "✅" if iter_val == rec_val else "❌"
    print(f"  {n}! = {rec_val:>6}   {match}")

# Tests rápidos con assert
assert factorial_recursivo(0) == 1
assert factorial_recursivo(5) == 120
assert factorial_recursivo(10) == 3628800
print("\nTodos los asserts pasaron ✅")


### 🔍 Traza de ejecución de `factorial_recursivo(4)`

```
factorial_recursivo(4)
  └─ 4 * factorial_recursivo(3)
         └─ 3 * factorial_recursivo(2)
                └─ 2 * factorial_recursivo(1)
                       └─ 1 * factorial_recursivo(0)
                              └─ retorna 1        ← caso base
                       retorna 1 * 1 = 1
                retorna 2 * 1 = 2
         retorna 3 * 2 = 6
  retorna 4 * 6 = 24
```

**Complejidad:**
- **Temporal**: O(n) — una llamada por cada valor desde n hasta 0
- **Espacial**: O(n) — n frames simultáneos en la pila de llamadas

---


---

# 📐 Sección 2: La Pila de Llamadas (Call Stack)

## Cómo gestiona Python las llamadas recursivas

Cada vez que una función se llama, Python crea un **frame** en la pila de llamadas con:
- Los parámetros locales
- Las variables locales
- La dirección de retorno

```
Stack (pila crece ↓)
┌─────────────────────────────┐
│  factorial_recursivo(0)  ← tope (se ejecuta primero en retornar)
├─────────────────────────────┤
│  factorial_recursivo(1)
├─────────────────────────────┤
│  factorial_recursivo(2)
├─────────────────────────────┤
│  factorial_recursivo(3)
├─────────────────────────────┤
│  factorial_recursivo(4)  ← fondo (llamada original)
└─────────────────────────────┘
```

**⚠️ Límite por defecto en Python**: `sys.getrecursionlimit()` = 1000

> Si la recursión supera ese límite, Python lanza `RecursionError: maximum recursion depth exceeded`


In [ ]:
# ─────────────────────────────────────────────────────────────
# Demostración de la pila de llamadas con decorador de traza
# ─────────────────────────────────────────────────────────────
import sys

def trazar(func):
    """Decorador que imprime cada llamada y retorno."""
    nivel = [0]
    def wrapper(*args):
        indent = "  " * nivel[0]
        print(f"{indent}→ {func.__name__}({', '.join(map(str, args))})")
        nivel[0] += 1
        result = func(*args)
        nivel[0] -= 1
        print(f"{indent}← {func.__name__}({', '.join(map(str, args))}) = {result}")
        return result
    return wrapper

@trazar
def factorial_trazado(n):
    if n == 0:
        return 1
    return n * factorial_trazado(n - 1)

print("Traza de factorial_trazado(4):")
print("─" * 40)
resultado = factorial_trazado(4)
print(f"\nResultado final: {resultado}")

print(f"\nLímite de recursión actual: {sys.getrecursionlimit()}")


---

# 🔢 Sección 3: Primeros Ejemplos Rigurosos

## Potencia entera, suma y dígitos

### Potencia x^n

Para calcular $x^n$ con $n \geq 0$:

$$
\text{power}(x, n) = \begin{cases}
1 & \text{si } n = 0 \\
x \cdot \text{power}(x, n-1) & \text{si } n > 0
\end{cases}
$$

**¿Podemos hacerlo más eficiente?** Sí: con *divide y conquista*:

$$
\text{power\_rapida}(x, n) = \begin{cases}
1 & \text{si } n = 0 \\
[\text{power}(x, n/2)]^2 & \text{si } n \text{ es par} \\
x \cdot [\text{power}(x, (n-1)/2)]^2 & \text{si } n \text{ es impar}
\end{cases}
$$

Esto reduce la complejidad de **O(n) → O(log n)**.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Potencia entera: O(n) vs O(log n)
# ─────────────────────────────────────────────────────────────

def power(x: float, n: int) -> float:
    """Potencia recursiva simple. O(n) tiempo, O(n) espacio."""
    if n == 0:
        return 1
    return x * power(x, n - 1)

def power_fast(x: float, n: int) -> float:
    """
    Potencia recursiva rápida (divide y conquista).
    T(n) = T(n/2) + O(1)  →  O(log n) tiempo, O(log n) espacio.
    """
    if n == 0:
        return 1
    parcial = power_fast(x, n // 2)
    resultado = parcial * parcial
    if n % 2 == 1:           # n impar: un factor extra
        resultado *= x
    return resultado

# Suma de los primeros n enteros: 1 + 2 + ... + n
def suma_enteros(n: int) -> int:
    """Caso base: n == 0. Caso recursivo: n + suma(n-1). O(n)."""
    if n == 0:
        return 0
    return n + suma_enteros(n - 1)

# Suma de dígitos
def suma_digitos(n: int) -> int:
    """Devuelve la suma de los dígitos de n >= 0. O(log n)."""
    if n < 10:
        return n
    return n % 10 + suma_digitos(n // 10)

# Demostración y validación
print("Potencia rápida:")
for base, exp in [(2, 10), (3, 5), (10, 6)]:
    rápido = power_fast(base, exp)
    lento  = power(base, exp)
    print(f"  {base}^{exp} = {rápido}   ({'✅' if rápido == lento else '❌'})")

print("\nSuma de enteros:")
for n in [0, 1, 5, 10]:
    fórmula = n * (n + 1) // 2
    print(f"  suma({n}) = {suma_enteros(n)}   (fórmula: {fórmula}) {'✅' if suma_enteros(n) == fórmula else '❌'}")

print("\nSuma de dígitos:")
for n in [0, 9, 123, 4567]:
    print(f"  suma_digitos({n}) = {suma_digitos(n)}")

# Asserts
assert power_fast(2, 10) == 1024
assert suma_enteros(10) == 55
assert suma_digitos(4567) == 22
print("\nTodos los asserts pasaron ✅")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Comparación de tiempos: O(n) vs O(log n) en potencia
# ─────────────────────────────────────────────────────────────
import time

def medir(func, *args, repeticiones=1000):
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        func(*args)
    return (time.perf_counter() - t0) / repeticiones * 1e6  # µs

ns = [10, 100, 500]
print(f"{'n':>6} | {'power O(n) µs':>14} | {'power_fast O(log n) µs':>22}")
print("-" * 50)
for n in ns:
    t_lento = medir(power, 2, n)
    t_rapido = medir(power_fast, 2, n)
    print(f"{n:>6} | {t_lento:>14.3f} | {t_rapido:>22.3f}")


---

## 🔍 Errores comunes con la recursión

| Error | Descripción | Ejemplo |
|-------|-------------|---------|
| **Caso base faltante** | La función nunca para | `def f(n): return n * f(n-1)` |
| **Caso base incorrecto** | Retorna valor equivocado | `if n == 0: return 0` en factorial |
| **No decremente hacia el caso base** | Siempre se llama con el mismo n | `return n * f(n)` |
| **Recursión de muy alta profundidad** | RecursionError | Fibonacci(1000) sin memoización |

### Cómo detectarlos

```python
# Ver el límite actual
import sys
print(sys.getrecursionlimit())  # 1000 por defecto

# Para estructuras muy grandes (con cuidado):
sys.setrecursionlimit(5000)
```


In [ ]:
# ─────────────────────────────────────────────────────────────
# Clase RecursionTrace — instrumento pedagógico para tracing
# ─────────────────────────────────────────────────────────────

class RecursionTrace:
    """
    Wrapper que traza automáticamente llamadas recursivas.

    Uso:
        @RecursionTrace.decorar
        def mi_funcion(n): ...

    Atributos:
        max_depth: profundidad máxima observada
        call_count: número total de llamadas
    """

    def __init__(self, func):
        self.func = func
        self._depth = 0
        self.max_depth = 0
        self.call_count = 0

    def __call__(self, *args):
        self._depth += 1
        self.call_count += 1
        self.max_depth = max(self.max_depth, self._depth)
        indent = "  " * (self._depth - 1)
        print(f"{indent}→ {self.func.__name__}({', '.join(map(str, args))})")
        result = self.func(*args)
        print(f"{indent}← {result}")
        self._depth -= 1
        return result

    @staticmethod
    def decorar(func):
        return RecursionTrace(func)

# Ejemplo de uso: factorial trazado con la clase
@RecursionTrace.decorar
def fact(n):
    if n == 0:
        return 1
    return n * fact(n - 1)

print("Traza completa de fact(5):")
print("─" * 35)
val = fact(5)
print(f"\nResultado: {val}")
print(f"Profundidad máxima: {fact.max_depth}")
print(f"Llamadas totales:   {fact.call_count}")


---

## 🌟 Resumen: Fundamentos de Recursión

| Concepto | Descripción clave |
|----------|------------------|
| **Caso base** | Detiene la recursión; devuelve resultado directo |
| **Caso recursivo** | Reduce el problema hacia el caso base |
| **Pila de llamadas** | n llamadas activas → O(n) espacio |
| **power_fast** | Un frame por bit de n → O(log n) espacio |
| **RecursionError** | Ocurre cuando se supera `sys.getrecursionlimit()` |

### 💡 Regla de oro del diseño recursivo

> 1. Define el **caso base** (el más simple posible)
> 2. Asume que la función funciona para n-1 *(hipótesis inductiva)*
> 3. Usa ese resultado para resolver n


In [ ]:
# Mini autoevaluación: ¿qué devuelve cada llamada?
# Antes de ejecutar, piensa en el resultado

def misterio(n):
    if n <= 0:
        return 0
    return n + misterio(n - 2)

# ¿Qué hace misterio? ¿Es par o impar dependiente?
for n in range(7):
    print(f"  misterio({n}) = {misterio(n)}")

print()
print("💭 Pregunta: ¿qué calcula misterio(n)?")
print("   (Observa el patrón: 0,1,2,4,6,9,12,...)")


In [ ]:
# Visualización: número de llamadas de power vs power_fast
import math

ns_plot = list(range(1, 51))
llamadas_lento  = [n for n in ns_plot]
llamadas_rapido = [math.ceil(math.log2(n + 1)) + 1 for n in ns_plot]

print(f"{'n':>4} | {'O(n) llamadas':>14} | {'O(log n) llamadas':>18}")
print("-" * 42)
for n, lento, rapido in zip(ns_plot[::5], llamadas_lento[::5], llamadas_rapido[::5]):
    print(f"{n:>4} | {lento:>14} | {rapido:>18}")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(ns_plot, llamadas_lento, label='power O(n)', lw=2)
    plt.plot(ns_plot, llamadas_rapido, label='power_fast O(log n)', lw=2, linestyle='--')
    plt.xlabel('n'); plt.ylabel('Llamadas recursivas')
    plt.title('Comparación de llamadas: power vs power_fast')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("(matplotlib no disponible — tabla arriba)")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Suite de tests — valida todo lo implementado en este notebook
# ─────────────────────────────────────────────────────────────
import unittest

class TestFundamentosRecursion(unittest.TestCase):

    def test_factorial(self):
        self.assertEqual(factorial_recursivo(0), 1)
        self.assertEqual(factorial_recursivo(1), 1)
        self.assertEqual(factorial_recursivo(5), 120)
        self.assertEqual(factorial_recursivo(10), 3628800)

    def test_factorial_iterativo_igual_recursivo(self):
        for n in range(12):
            self.assertEqual(factorial_iterativo(n), factorial_recursivo(n))

    def test_power(self):
        self.assertEqual(power(2, 0), 1)
        self.assertEqual(power(2, 10), 1024)
        self.assertAlmostEqual(power(1.5, 4), 5.0625)

    def test_power_fast_same_as_power(self):
        for base in [2, 3, 5]:
            for exp in range(15):
                self.assertEqual(power(base, exp), power_fast(base, exp))

    def test_suma_enteros(self):
        self.assertEqual(suma_enteros(0), 0)
        self.assertEqual(suma_enteros(10), 55)
        self.assertEqual(suma_enteros(100), 5050)

    def test_suma_digitos(self):
        self.assertEqual(suma_digitos(0), 0)
        self.assertEqual(suma_digitos(9), 9)
        self.assertEqual(suma_digitos(123), 6)
        self.assertEqual(suma_digitos(9999), 36)

runner = unittest.TextTestRunner(verbosity=2)
suite  = unittest.TestLoader().loadTestsFromTestCase(TestFundamentosRecursion)
runner.run(suite)


---

## 🌟 Recursos del capítulo

### 📚 Lectura obligatoria
- Goodrich, Tamassia & Goldwasser — **Capítulo 4: Recursion** (secciones 4.1–4.2)

### 🌐 Herramientas interactivas
- **Python Tutor** (pythontutor.com) — visualiza la pila de llamadas paso a paso
- **Recursion Visualizer** (recursion.vercel.app) — árbol de llamadas animado

### 🔗 Práctica adicional
- GeeksforGeeks Recursion: https://www.geeksforgeeks.org/recursion/
- LeetCode Recursion I: https://leetcode.com/explore/learn/card/recursion-i/

---

## 📊 Tabla resumen de complejidad

| Función | T(n) | S(n) | Tipo de recursión |
|---------|------|------|-------------------|
| `factorial_recursivo(n)` | O(n) | O(n) | Lineal |
| `power(x, n)` | O(n) | O(n) | Lineal |
| `power_fast(x, n)` | O(log n) | O(log n) | Lineal (divide y conquista) |
| `suma_enteros(n)` | O(n) | O(n) | Lineal |
| `suma_digitos(n)` | O(log n) | O(log n) | Lineal |


---

## 🧠 Autoevaluación (responder antes de la próxima clase)

1. ¿Cuál es la diferencia entre **caso base** y **caso recursivo**? Da un ejemplo de cada uno.
2. ¿Qué ocurre si escribes `factorial_recursivo(-1)`? ¿Cómo lo corregirías?
3. ¿Por qué `power_fast(2, n)` necesita O(log n) frames en la pila y no O(n)?
4. Escribe la traza completa de `suma_enteros(3)` a mano (sin ejecutar).
5. ¿En qué situaciones preferirías la versión iterativa sobre la recursiva?

---

## 🔗 Conexión con el tema siguiente

En el **Notebook 02**, exploraremos algoritmos recursivos clásicos más complejos:
- Búsqueda binaria recursiva (O(log n), el problema más elegante)
- Exploración del sistema de archivos (árbol de directorios)
- Torres de Hanói (O(2^n) — cuándo la recursión tiene costo alto)
- Fibonacci: de O(2^n) a O(n) con memoización


---

*Estructuras de Datos y Algoritmos — Lic. en Sistemas | Capítulo 4: Recursión*
*Notebook 01: Fundamentos de Recursión | v1.0 — Marzo 2026*
